In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import csv

# Basispfad definieren
BASE_DIR = r"C:\Users\flori\Desktop\output_scenarios_test\Test Bay"
U_MAX_THRESHOLD = 253.0 # Grenzwert für Spannung (230V + 10%)

# Fenstergröße für die Rolling Features ---
# Bei einem 15-Minuten-Messtakt entsprechen 4 Zeilen genau 1 Stunde.
WINDOW_SIZE = 4 

def process_file(file_path, prefix):
    # 1. Datei exakt im Originalformat einlesen
    df = pd.read_csv(file_path, sep=',', decimal=',')
    
    # 2. Relevante Spalten für die Berechnung identifizieren
    u_cols = [c for c in df.columns if 'Spannung' in c and ('U1-N' in c or 'L1N' in c)]
    p_cols = [c for c in df.columns if 'Wirkleistung' in c and 'gesamt' in c]
    pf_cols = [c for c in df.columns if ('Powerfactor' in c or 'Leistungsfaktor' in c) and 'gesamt' in c]

    # Falls die Datei nicht die erwarteten Spalten hat, unverändert zurückgeben
    if not u_cols or not p_cols:
        return df
        
    u_col = u_cols[0]
    p_col = p_cols[0]

    # Temporär für die Berechnung sicherstellen, dass die Werte numerisch sind
    u_series = pd.to_numeric(df[u_col], errors='coerce')
    p_series = pd.to_numeric(df[p_col], errors='coerce')
    
    # --- 3. Physikalische ROLLING Features berechnen ---
    # Wir erstellen "Rolling-Objekte", die über die Zeitreihe gleiten
    u_rolling = u_series.rolling(window=WINDOW_SIZE, min_periods=1)
    p_rolling = p_series.rolling(window=WINDOW_SIZE, min_periods=1)
    
    # Werte als neue Spalten direkt in den DataFrame schreiben
    df[f'{prefix}Spannungshub_rolling'] = u_rolling.max() - u_rolling.min()
    
    # Varianz benötigt mindestens 2 Werte, daher füllen wir NaN in der ersten Zeile mit 0
    df[f'{prefix}Spannung_Varianz_rolling'] = u_rolling.var().fillna(0) 
    
    df[f'{prefix}P_max_rolling'] = p_rolling.max()
    df[f'{prefix}P_min_rolling'] = p_rolling.min()
    
    # Grenzwertverletzungen: Wahr/Falsch in 1/0 umwandeln und dann summieren
    is_over_threshold = (u_series > U_MAX_THRESHOLD).astype(int)
    df[f'{prefix}Grenzwertverletzungen_rolling'] = is_over_threshold.rolling(window=WINDOW_SIZE, min_periods=1).sum()
    
    # Cos Phi als gleitender Mittelwert
    if pf_cols:
        pf_series = pd.to_numeric(df[pf_cols[0]], errors='coerce')
        df[f'{prefix}CosPhi_Mittel_rolling'] = pf_series.rolling(window=WINDOW_SIZE, min_periods=1).mean()

    return df

def run_extraction():
    # Hier werden die exakten gewünschten Präfixe zugewiesen
    folders = {
        "Test_Bay_F1": "smartmeter_",
        "Test_Bay_F2": "ortsnetztrafo_"
    }

    for folder_name, prefix in folders.items():
        search_path = os.path.join(BASE_DIR, folder_name, "Extracted_Measurements", "*.csv")
        files = glob.glob(search_path)
        
        if not files:
            print(f"Keine Dateien im Ordner {folder_name} gefunden.")
            continue
            
        # Neuen Unterordner für die Ergebnisse erstellen
        output_folder = os.path.join(BASE_DIR, folder_name, "Extracted_Measurements", "Processed_with_Rolling_Features")
        os.makedirs(output_folder, exist_ok=True)
        
        for f in files:
            filename = os.path.basename(f)
            print(f"Verarbeite: {filename}...")
            
            try:
                # Datenverarbeitung aufrufen
                processed_df = process_file(f, prefix)
                
                # Speichern im exakt gleichen Format wie die Originaldatei!
                output_path = os.path.join(output_folder, filename)
                processed_df.to_csv(output_path, index=False, sep=',', decimal=',', quoting=csv.QUOTE_MINIMAL)
                
            except Exception as e:
                print(f"Fehler bei Datei {filename}: {e}")

if __name__ == "__main__":
    print("Starte Datenverarbeitung mit Rolling Features...")
    run_extraction()
    print("\nErfolgreich beendet! Die Originalstruktur wurde beibehalten.")

Starte Datenverarbeitung mit Rolling Features...
Verarbeite: output_10_A_c.csv...
Verarbeite: output_10_A_w.csv...
Verarbeite: output_11_A_c.csv...
Verarbeite: output_11_A_w.csv...
Verarbeite: output_12_A_c.csv...
Verarbeite: output_12_A_w.csv...
Verarbeite: output_13_A_c.csv...
Verarbeite: output_13_A_w.csv...
Verarbeite: output_14_A_c.csv...
Verarbeite: output_14_A_w.csv...
Verarbeite: output_15_A_c.csv...
Verarbeite: output_15_A_w.csv...
Verarbeite: output_16_A_c.csv...
Verarbeite: output_16_A_w.csv...
Verarbeite: output_17_A_c.csv...
Verarbeite: output_17_A_w.csv...
Verarbeite: output_18_A_c.csv...
Verarbeite: output_18_A_w.csv...
Verarbeite: output_19_A_c.csv...
Verarbeite: output_19_A_w.csv...
Verarbeite: output_1_A_c.csv...
Verarbeite: output_1_A_w.csv...
Verarbeite: output_2_A_c.csv...
Verarbeite: output_2_A_w.csv...
Verarbeite: output_3_A_c.csv...
Verarbeite: output_3_A_w.csv...
Verarbeite: output_4_A_c.csv...
Verarbeite: output_4_A_w.csv...
Verarbeite: output_5_A_c.csv...
Ver